# Конджойнт-эксперимент

In [1]:
import random
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict, Counter
from scipy.special import expit, logit
from patsy import dmatrix
import statsmodels.formula.api as smf
from linearmodels import OLS

random.seed(42)
np.random.seed(42)

In [84]:
# параметры симуляции
n_simulations = 2000
alpha = 0.05
levels = [7, 4, 3, 2, 2, 9, 4, 2, 4, 2] # число уровней по каждому атрибуту

# Симуляция мощности

In [85]:
def compute_power(n_respondents, n_tasks, amce, levels, n_simulations, alpha=0.05, seed=42):
    ### Симулирует эксперимент n_simulations раз и считает долю случаев,
    ### когда AMCE оказался статистически значимым - мощность
    rng = np.random.default_rng(seed)
    n_significant = 0

    for i in range(n_simulations):
        choices = []
        dummies = []

        for i in range(n_respondents * n_tasks):
            # случайно генерируем два профиля
            profile_a = [rng.integers(0, L) for L in levels]
            profile_b = [rng.integers(0, L) for L in levels]

            # вероятность выбрать A зависит от того, у кого нужный уровень первого атрибута
            prob = 0.5 + amce * (int(profile_a[0] == 1) - int(profile_b[0] == 1))
            prob = np.clip(prob, 0, 1)

            choices.append(int(rng.random() < prob))
            dummies.append(int(profile_a[0] == 1) - int(profile_b[0] == 1))

        y = np.array(choices, dtype=float)
        x = np.array(dummies,  dtype=float)

        denom = ((x - x.mean()) ** 2).sum()
        if denom == 0:
            continue

        # оцениваем AMCE
        beta = ((x - x.mean()) * (y - y.mean())).sum() / denom
        resid = y - (beta * x + y.mean() - beta * x.mean())
        se = np.sqrt((resid ** 2).sum() / (len(y) - 2) / denom)

        if se > 0 and abs(beta / se) > 1.96:
            n_significant += 1

    return n_significant / n_simulations


print(f"{'N':>5}  {'tasks':>6}  {'power':>8}")
for n in [500, 600, 700, 800]:
    power = compute_power(n, 15, 0.05, levels, n_simulations)
    print(f"{n:>5}  {'15':>6}  {power:.3f}")

    N   tasks     power
  500      15  0.983
  600      15  0.996
  700      15  0.999
  800      15  1.000


# Атрибуты и параметры

In [86]:
attributes = {
    'region': ['Восточная Азия (Китай, Южная Корея, Монголия)', 'Средняя Азия (Узбекистан, Таджикистан, Кыргызстан)',
        'Африка (Нигерия, ЮАР, Кения)', 'Южный Кавказ (Армения, Азербайджан, Грузия)',
        'Восточная Европа (Беларусь, Молдова, Украина)', 'Юго-Восточная Европа (Венгрия, Сербия, Болгария)', 
               'Западная Европа (Германия, Франция, Италия)'],
    'motivation': ['Поиск работы', 'Воссоединение с супругом(ой), ранее приехавшим(ей) в Россию',
                   'Бегство от вооружённого конфликта', 'Бегство от политических преследований'],
    'education': ['Среднее (школа)', 'Среднее специальное (колледж)', 'Высшее (университет)'],
    'employer': ['Государственная организация', 'Небольшая частная компания'],
    'gender': ['Мужчина','Женщина'],
    'age': [21,22,23, 45, 46, 47, 61, 62, 63],
    'occupation': ['Врач', 'Программист', 'Строитель', 'Сфера услуг (общепит)'],
    'language': ['Говорит свободно','Говорит плохо'],
    'politics': ['Страна помогает России обходить санкции (параллельный экспорт)',
        'Страна обычно голосует вместе с Россией в Генассамблее ООН', 'Страна обычно голосует вместе с США в Генассамблее ООН',
        'Страна присоединилась к санкциям против России'],
    'appearance': ['В повседневной жизни носит традиционную национальную одежду','В повседневной жизни носит обычную повседневную одежду'
    ],
}

n_respondents = 800
n_tasks = 15

# сколько параметров b надо оценить в модели (для каждого атрибута — число уровней минус 1)
n_params = sum(len(v) - 1 for v in attributes.values())
print(f'Параметров в модели: {n_params}')
print(f'Всего оценок от респондентов: {n_respondents * n_tasks}')

Параметров в модели: 29
Всего оценок от респондентов: 12000


# Генерация реалистичных профилей  
Сначала генерирую все возможные комбинации атрибутов, потом убираю нереалистичные. Нереалистичные - это те, где комбинация атрибутов не встречается в реальной жизни.

In [87]:
def is_realistic(profile):
    region = profile['region']
    motivation  = profile['motivation']
    education = profile['education']
    age  = profile['age']
    occupation = profile['occupation']
    politics = profile['politics']

    # Регионы
    WEST_EU = 'Западная Европа (Германия, Франция, Италия)'
    SE_EU = 'Юго-Восточная Европа (Венгрия, Сербия, Болгария)'
    E_EU = 'Восточная Европа (Беларусь, Молдова, Украина)'
    CAUC = 'Южный Кавказ (Армения, Азербайджан, Грузия)'
    CA = 'Средняя Азия (Узбекистан, Таджикистан, Кыргызстан)'
    AFR  = 'Африка (Нигерия, ЮАР, Кения)'
    EA = 'Восточная Азия (Китай, Южная Корея, Монголия)'

    EUROPE_ALL = (WEST_EU, SE_EU, E_EU)

    # Политика
    PAR_EXPORT = 'Страна помогает России обходить санкции (параллельный экспорт)'
    SANCTIONS  = 'Страна присоединилась к санкциям против России'
    ARMED_CONFLICT = 'Бегство от вооружённого конфликта'
    if region in EUROPE_ALL and politics == PAR_EXPORT:
        return False
    if region == CA and politics == SANCTIONS:
        return False
    if occupation == 'Врач' and age == 21:
        return False
    if occupation == 'Врач' and age == 22:
        return False
    if occupation == 'Врач' and education != 'Высшее (университет)':
        return False
    if motivation == ARMED_CONFLICT and region in (
        WEST_EU
    ):
        return False

    return True

all_profiles = [
    dict(zip(attributes.keys(), combo))
    for combo in product(*attributes.values())
]

realistic = [p for p in all_profiles if is_realistic(p)]
profiles  = pd.DataFrame(realistic).reset_index(drop=True)

print(f'Всего профилей:{len(all_profiles)}')
print(f'Нереалистичных: {len(all_profiles) - len(realistic)}')
print(f'Реалистичных: {len(profiles)}')
print(f'Возможных пар: {len(profiles) * (len(profiles) - 1) // 2:,}')

Всего профилей:193536
Нереалистичных: 62592
Реалистичных: 130944
Возможных пар: 8,573,100,096


In [122]:
# истинные вероятности уровней в генеральной совокупности
true_probs = {}
for attr in attributes:
    counts = profiles[attr].value_counts()
    true_probs[attr] = (counts / counts.sum()).to_dict()

for attr, probs in true_probs.items():
    print(f'\n{attr}')
    for lvl, p in sorted(probs.items(), key=lambda x: -x[1]):
        print(f'  {str(lvl):} {p:.4f}')


region
  Восточная Азия (Китай, Южная Корея, Монголия) 0.1720
  Африка (Нигерия, ЮАР, Кения) 0.1720
  Южный Кавказ (Армения, Азербайджан, Грузия) 0.1720
  Средняя Азия (Узбекистан, Таджикистан, Кыргызстан) 0.1290
  Восточная Европа (Беларусь, Молдова, Украина) 0.1290
  Юго-Восточная Европа (Венгрия, Сербия, Болгария) 0.1290
  Западная Европа (Германия, Франция, Италия) 0.0968

motivation
  Поиск работы 0.2581
  Воссоединение с супругом(ой), ранее приехавшим(ей) в Россию 0.2581
  Бегство от политических преследований 0.2581
  Бегство от вооружённого конфликта 0.2258

education
  Высшее (университет) 0.3864
  Среднее (школа) 0.3068
  Среднее специальное (колледж) 0.3068

employer
  Государственная организация 0.5000
  Небольшая частная компания 0.5000

gender
  Мужчина 0.5000
  Женщина 0.5000

age
  23 0.1136
  45 0.1136
  46 0.1136
  47 0.1136
  61 0.1136
  62 0.1136
  63 0.1136
  21 0.1023
  22 0.1023

occupation
  Программист 0.3068
  Строитель 0.3068
  Сфера услуг (общепит) 0.3068
 

# Генерация профилей

In [110]:
def deficit(i, total_counts):
    score = 0
    for attr in attributes:
        lvl = profiles.at[i, attr]
        n_seen = total_counts[(attr, lvl)]
        n_total = sum(total_counts[(attr, l)] for l in attributes[attr])
        observed = n_seen / n_total if n_total > 0 else 0
        score += true_probs[attr].get(lvl, 0) - observed
    return score


def sample_pairs(n_pairs, seed=42):
    random.seed(seed)
    idx = list(profiles.index)
    selected = set()
    total_counts = defaultdict(int)
    max_attempts = n_pairs * 500

    for attempt in range(max_attempts):
        if len(selected) >= n_pairs:
            break
        pool_1 = random.sample(idx, min(300, len(idx)))
        profile_1 = max(pool_1, key=lambda i: deficit(i, total_counts))

        pool_2 = random.sample(
            [i for i in idx if i != profile_1],
            min(150, len(idx) - 1)
        )
        profile_2 = None
        best = float('-inf')

        for candidate_idx in range(len(pool_2)):
            candidate = pool_2[candidate_idx]
            pair = (min(profile_1, candidate), max(profile_1, candidate))
            if pair in selected:
                continue
            # профили должны отличаться хотя бы по двум атрибутам
            n_diff = sum(
                profiles.at[profile_1, a] != profiles.at[candidate, a]
                for a in attributes
            )
            if n_diff < 2:
                continue
            score = deficit(candidate, total_counts)
            if score > best:
                best, profile_2 = score, candidate

        if profile_2 is None:
            continue

        pair = (min(profile_1, profile_2), max(profile_1, profile_2))
        selected.add(pair)
        for attr in attributes:
            total_counts[(attr, profiles.at[profile_1, attr])] += 1
            total_counts[(attr, profiles.at[profile_2, attr])] += 1

    return list(selected)

In [111]:
selected_pairs_300 = sample_pairs(n_pairs=300, seed=41)
print(len(selected_pairs_300))

300


In [112]:
rows_for_df = []
for i, j in selected_pairs_300:
    for idx in [i, j]:
        rows_for_df.append({a: profiles.at[idx, a] for a in attributes})
attr_df = pd.DataFrame(rows_for_df)

print('Баланс уровней по атрибутам:')
for attr in attributes:
    counts_attr = attr_df[attr].value_counts()
    cv = counts_attr.std() / counts_attr.mean() * 100
    for level, count in counts_attr.items():
        print(f'{str(level):} {count}')

Баланс уровней по атрибутам:
Африка (Нигерия, ЮАР, Кения) 103
Восточная Азия (Китай, Южная Корея, Монголия) 103
Южный Кавказ (Армения, Азербайджан, Грузия) 103
Юго-Восточная Европа (Венгрия, Сербия, Болгария) 78
Средняя Азия (Узбекистан, Таджикистан, Кыргызстан) 78
Восточная Европа (Беларусь, Молдова, Украина) 77
Западная Европа (Германия, Франция, Италия) 58
Воссоединение с супругом(ой), ранее приехавшим(ей) в Россию 156
Бегство от политических преследований 155
Поиск работы 154
Бегство от вооружённого конфликта 135
Высшее (университет) 232
Среднее (школа) 185
Среднее специальное (колледж) 183
Небольшая частная компания 301
Государственная организация 299
Женщина 300
Мужчина 300
62 69
47 69
46 69
45 68
23 68
63 68
61 67
22 61
21 61
Сфера услуг (общепит) 184
Строитель 184
Программист 184
Врач 48
Говорит свободно 301
Говорит плохо 299
Страна обычно голосует вместе с Россией в Генассамблее ООН 174
Страна обычно голосует вместе с США в Генассамблее ООН 174
Страна присоединилась к санкциям

In [113]:
# restricted randomization AMCE (хи-квадрат и нарушения независимости
# связаны с моими же ограничениями)
pairs_to_check = [
    ('region', 'politics'),
    ('region', 'motivation'),
    ('region', 'language'),
    ('region', 'appearance'),
    ('occupation', 'age'),
    ('occupation', 'education'),
    ('occupation', 'language'),
    ('motivation', 'occupation'),
    ('motivation', 'age'),
    ('politics', 'motivation'),
    ('education', 'age'),
]
for a1, a2 in pairs_to_check:
    ct = pd.crosstab(attr_df[a1], attr_df[a2])
    expected = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / ct.values.sum()
    chi2 = ((ct.values - expected) ** 2 / expected).sum()
    print(f'{a1} * {a2} chi2 = {chi2:.1f}')

region * politics chi2 = 129.1
region * motivation chi2 = 41.5
region * language chi2 = 8.3
region * appearance chi2 = 9.2
occupation * age chi2 = 51.5
occupation * education chi2 = 87.3
occupation * language chi2 = 3.3
motivation * occupation chi2 = 21.9
motivation * age chi2 = 24.9
politics * motivation chi2 = 15.2
education * age chi2 = 33.0


In [114]:
for a1, a2 in pairs_to_check:
    ct = pd.crosstab(attr_df[a1], attr_df[a2])
    expected = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / ct.values.sum()
    chi2 = ((ct.values - expected) ** 2 / expected).sum()
    n = ct.values.sum()
    v = (chi2 / (n * (min(ct.shape) - 1))) ** 0.5
    print(f'{a1} * {a2} chi2 = {chi2:.1f}, V = {v:.3f}')

region * politics chi2 = 129.1, V = 0.268
region * motivation chi2 = 41.5, V = 0.152
region * language chi2 = 8.3, V = 0.118
region * appearance chi2 = 9.2, V = 0.124
occupation * age chi2 = 51.5, V = 0.169
occupation * education chi2 = 87.3, V = 0.270
occupation * language chi2 = 3.3, V = 0.074
motivation * occupation chi2 = 21.9, V = 0.110
motivation * age chi2 = 24.9, V = 0.118
politics * motivation chi2 = 15.2, V = 0.092
education * age chi2 = 33.0, V = 0.166


In [117]:
rows_300 = []
for pair_id, (i, j) in enumerate(selected_pairs_300, 1):
    row = {'pair_id': pair_id}
    for attr in attributes:
        row[f'A_{attr}'] = profiles.at[i, attr]
        row[f'B_{attr}'] = profiles.at[j, attr]
    rows_300.append(row)

pairs_df_300 = pd.DataFrame(rows_300)
pairs_df_300.to_excel('conjoint_pairs_300_final.xlsx', index=False)

# Симуляция

In [118]:
target_amce = {
    'region_Юго-Восточная Европа (Венгрия, Сербия, Болгария)': -0.02,
    'region_Западная Европа (Германия, Франция, Италия)': -0.03,
    'region_Южный Кавказ (Армения, Азербайджан, Грузия)': -0.08,
    'region_Восточная Азия (Китай, Южная Корея, Монголия)': -0.12,
    'region_Средняя Азия (Узбекистан, Таджикистан, Кыргызстан)': -0.15,
    'region_Африка (Нигерия, ЮАР, Кения)': -0.18,
    'motivation_Воссоединение с супругом(ой), ранее приехавшим(ей) в Россию': 0.03,
    'motivation_Бегство от вооружённого конфликта': 0.05,
    'motivation_Бегство от политических преследований': 0.06,
    'education_Среднее специальное (колледж)': 0.05,
    'education_Высшее (университет)': 0.10,
    'employer_Небольшая частная компания': -0.05,
    'gender_Женщина': 0.05,
    'age_45': -0.03,
    'age_62': -0.07,
    'occupation_Программист': -0.03,
    'occupation_Строитель': -0.10,
    'occupation_Сфера услуг (общепит)': -0.12,
    'language_Говорит плохо': -0.18,
    'politics_Страна обычно голосует вместе с Россией в Генассамблее ООН': -0.02,
    'politics_Страна обычно голосует вместе с США в Генассамблее ООН': -0.08,
    'politics_Страна присоединилась к санкциям против России': -0.12,
    'appearance_В повседневной жизни носит традиционную национальную одежду': -0.08,
}

# перевод AMCE в коэффициенты логит-модели
coefficients = {k: logit(0.5 + v) for k, v in target_amce.items()}

In [119]:
def compute_utility(prefix, row):
    return sum(coefficients.get(f'{attr}_{row[f"{prefix}_{attr}"]}', 0.0) for attr in attrs)


def simulate_experiment(pairs, n_respondents, n_tasks, seed=42):
    np.random.seed(seed)
    rows = []

    for respondent_id in range(n_respondents):
        block_id = respondent_id % (len(pairs) // n_tasks)
        for i, pair in pairs.sample(n=n_tasks, replace=False).iterrows():
            p_a    = expit(compute_utility('A', pair) - compute_utility('B', pair))
            choice = int(p_a > 0.5)
            for prefix, c in [('A', choice), ('B', 1 - choice)]:
                rows.append(
                    {a: pair[f'{prefix}_{a}'] for a in attrs} |
                    {'choice': c, 'respondent_id': respondent_id,
                     'block_id': block_id, 'age': str(pair[f'{prefix}_age'])}
                )
    return pd.DataFrame(rows)

In [120]:
attrs = [c[2:] for c in pairs_df_300.columns if c.startswith('A_')]
experiment_300 = simulate_experiment(pairs_df_300, n_respondents, n_tasks)

formula = (
    'choice ~ '
    'C(region, Treatment("Восточная Европа (Беларусь, Молдова, Украина)")) + '
    'C(motivation, Treatment("Поиск работы")) + '
    'C(education, Treatment("Среднее (школа)")) + '
    'C(employer, Treatment("Государственная организация")) + '
    'C(gender, Treatment("Мужчина")) + '
    'C(age, Treatment("21")) + '
    'C(occupation, Treatment("Врач")) + '
    'C(language, Treatment("Говорит свободно")) + '
    'C(politics, Treatment("Страна помогает России обходить санкции (параллельный экспорт)")) + '
    'C(appearance, Treatment("В повседневной жизни носит обычную повседневную одежду"))'
)

model_300 = smf.ols(formula, data=experiment_300).fit(cov_type='cluster', cov_kwds={'groups': experiment_300['respondent_id']}
)

print(model_300.summary())

                            OLS Regression Results                            
Dep. Variable:                 choice   R-squared:                       0.253
Model:                            OLS   Adj. R-squared:                  0.252
Method:                 Least Squares   F-statistic:                     616.5
Date:                Fri, 24 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:26:39   Log-Likelihood:                -13919.
No. Observations:               24000   AIC:                         2.790e+04
Df Residuals:                   23970   BIC:                         2.814e+04
Df Model:                          29                                         
Covariance Type:              cluster                                         
                                                                                                                                                             coef    std err          z      P>|z|      [0.025    

In [121]:
pairs_pool = pairs_df_300.rename(columns={
    f'{prefix}_{attr}': f'profile_{prefix}_{attr}'
    for prefix in ['A', 'B'] for attr in attributes
})
for col in ['A_age', 'B_age']:
    pairs_df_300[col] = pairs_df_300[col].astype(str)